In [1]:
!pip install pyspark

In [2]:
import sys
sys.executable

'/home/smmash/myenv/bin/python'

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [4]:
# Build the SparkSession with proper syntax and quoting
spark = (SparkSession.builder 
         .appName("Airbnb Analysis") 
         .config("spark.sql.shuffle.partitions", 4) 
         .getOrCreate())

# Set log level using a string
spark.sparkContext.setLogLevel("ERROR")

# Print version with a correctly formatted f-string
print(f"Spark Version: {spark.version}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/07 09:49:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Version: 3.5.1


### Loading Data

In [8]:
listings_df = spark.read.csv(
    "Data/listings.csv.gz",
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"'
)

reviews_df = spark.read.csv(
    "Data/reviews.csv.gz",
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"'
)

calendar_df = spark.read.csv(
    "Data/calendar.csv.gz",
    header=True,
    inferSchema=True
)

print(f"listings-> {listings_df.count()} rows, {len(listings_df.columns)} columns")
print(f"Reviews-> {reviews_df.count()} rows, {len(reviews_df.columns)} columns")
print(f"Calendar-> {calendar_df.count()} rows, {len(calendar_df.columns)} columns")

    
    

listings-> 96871 rows, 79 columns


Reviews-> 2097996 rows, 6 columns


[Stage 20:>                                                         (0 + 1) / 1]

Calendar-> 35357974 rows, 7 columns


### Exploring DataFrames

In [9]:
print("Listings Schema: ")
listings_df.printSchema()

Listings Schema: 
root
 |-- id: long (nullable = true)
 |-- listing_url: string (nullable = true)
 |-- scrape_id: long (nullable = true)
 |-- last_scraped: date (nullable = true)
 |-- source: string (nullable = true)
 |-- name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- neighborhood_overview: string (nullable = true)
 |-- picture_url: string (nullable = true)
 |-- host_id: integer (nullable = true)
 |-- host_url: string (nullable = true)
 |-- host_name: string (nullable = true)
 |-- host_since: date (nullable = true)
 |-- host_location: string (nullable = true)
 |-- host_about: string (nullable = true)
 |-- host_response_time: string (nullable = true)
 |-- host_response_rate: string (nullable = true)
 |-- host_acceptance_rate: string (nullable = true)
 |-- host_is_superhost: string (nullable = true)
 |-- host_thumbnail_url: string (nullable = true)
 |-- host_picture_url: string (nullable = true)
 |-- host_neighbourhood: string (nullable = true)
 |-- host_l

In [10]:
print("Reviews Schema: ")
reviews_df.printSchema()

Reviews Schema: 
root
 |-- listing_id: long (nullable = true)
 |-- id: long (nullable = true)
 |-- date: date (nullable = true)
 |-- reviewer_id: integer (nullable = true)
 |-- reviewer_name: string (nullable = true)
 |-- comments: string (nullable = true)



In [11]:
print("Date Schema: ")
calendar_df.printSchema()

Date Schema: 
root
 |-- listing_id: long (nullable = true)
 |-- date: date (nullable = true)
 |-- available: string (nullable = true)
 |-- price: string (nullable = true)
 |-- adjusted_price: string (nullable = true)
 |-- minimum_nights: integer (nullable = true)
 |-- maximum_nights: integer (nullable = true)



In [13]:
listings_df.show(5, truncate=False)

+-----+--------------------+--------------+------------+---------------+--------------------+--------------------+---------------------+--------------------+-------+--------------------+---------+----------+--------------------+--------------------+------------------+------------------+--------------------+-----------------+--------------------+--------------------+------------------+-------------------+-------------------------+--------------------+--------------------+----------------------+--------------------+----------------------+----------------------------+--------+---------+--------------------+---------------+------------+---------+--------------+--------+----+--------------------+-------+--------------+--------------+----------------------+----------------------+----------------------+----------------------+----------------------+----------------------+----------------+----------------+---------------+---------------+---------------+----------------+---------------------+---

In [14]:
reviews_df.show(5, truncate=True)

+----------+------+----------+-----------+-------------+--------------------+
|listing_id|    id|      date|reviewer_id|reviewer_name|            comments|
+----------+------+----------+-----------+-------------+--------------------+
|     13913| 80770|2010-08-18|     177109|      Michael|My girlfriend and...|
|     13913|367568|2011-07-11|   19835707|      Mathias|Alina was a reall...|
|     13913|529579|2011-09-13|    1110304|      Kristin|Alina is an amazi...|
|     13913|595481|2011-10-03|    1216358|      Camilla|Alina's place is ...|
|     13913|612947|2011-10-09|     490840|        Jorik|Nice location in ...|
+----------+------+----------+-----------+-------------+--------------------+
only showing top 5 rows



In [17]:
calendar_df.show(5)

+-------------------+----------+---------+-----+--------------+--------------+--------------+
|         listing_id|      date|available|price|adjusted_price|minimum_nights|maximum_nights|
+-------------------+----------+---------+-----+--------------+--------------+--------------+
|1196288722069341420|2025-09-15|        f| NULL|          NULL|             1|          1125|
|1196288722069341420|2025-09-16|        f| NULL|          NULL|             1|          1125|
|1196288722069341420|2025-09-17|        f| NULL|          NULL|             1|          1125|
|1196288722069341420|2025-09-18|        f| NULL|          NULL|             1|          1125|
|1196288722069341420|2025-09-19|        f| NULL|          NULL|             1|          1125|
+-------------------+----------+---------+-----+--------------+--------------+--------------+
only showing top 5 rows



In [18]:
# Select few important numeric columns for "describe"
listings_df.select(
    "price", "minimum_nights", "number_of_reviews",
    "review_scores_rating", "calculated_host_listings_count"
).describe().show()               # "describe()" function figure out count, mean, stddev, min, max

[Stage 28:>                                                         (0 + 1) / 1]

+-------+---------+------------------+------------------+--------------------+------------------------------+
|summary|    price|    minimum_nights| number_of_reviews|review_scores_rating|calculated_host_listings_count|
+-------+---------+------------------+------------------+--------------------+------------------------------+
|  count|    61963|             96871|             96871|               72749|                         96871|
|   mean|     NULL|5.4402968896780255| 21.65762715363731|   4.684718827750277|            16.685499272228014|
| stddev|     NULL|23.686680781928313|50.368643995520806| 0.49419100920851045|             53.13028957467019|
|    min|$1,000.00|                 1|                 0|                 0.0|                             1|
|    max|  $999.00|              1125|              1902|                 5.0|                           500|
+-------+---------+------------------+------------------+--------------------+------------------------------+



In [20]:
listings_df.select(
    "id", "name", "host_name", "neighbourhood_cleansed",
    "room_type", "price", "number_of_reviews"
).show(10, truncate=40)

+-----+----------------------------------------+------------+----------------------+---------------+-------+-----------------+
|   id|                                    name|   host_name|neighbourhood_cleansed|      room_type|  price|number_of_reviews|
+-----+----------------------------------------+------------+----------------------+---------------+-------+-----------------+
|13913|     Holiday London DB Room Let-on going|       Alina|             Islington|   Private room| $70.00|               55|
|15400|     Bright Chelsea  Apartment. Chelsea!|    Philippa|Kensington and Chelsea|Entire home/apt|$149.00|               97|
|17402|Very Central Modern 3-Bed/2 Bath By O...|         Liz|           Westminster|Entire home/apt|$411.00|               56|
|24328|        Battersea live/work artist house|         Joe|            Wandsworth|Entire home/apt|   NULL|               95|
|36274|Bright 1 bedroom apt off brick lane i...|    Hendryks|         Tower Hamlets|Entire home/apt|$210.00|   

In [21]:
# Listings with more than 100 reviews
popular = listings_df.filter(F.col("number_of_reviews") > 100)
print(f"Listings with > 100 reviews: {popular.count():}")
popular.select("name", "number_of_reviews", "room_type").show(10)

[Stage 33:>                                                         (0 + 1) / 1]

Listings with > 100 reviews: 4643
+--------------------+-----------------+---------------+
|                name|number_of_reviews|      room_type|
+--------------------+-----------------+---------------+
|Kew Gardens 3BR h...|              116|Entire home/apt|
|You are GUARANTEE...|              730|   Private room|
|SUNNY ROOM PRIVAT...|              387|   Private room|
|Room with a view,...|              137|   Private room|
|You Will Save Mon...|              639|   Private room|
|Quiet Comfortable...|              266|   Private room|
|Beautiful 1 bed a...|              145|Entire home/apt|
|Best Part of Town...|              101|Entire home/apt|
|Beautiful, Luxuri...|              224|   Private room|
|Pleasant Single R...|              456|   Private room|
+--------------------+-----------------+---------------+
only showing top 10 rows



In [25]:
# Count distinct values
print("Distinct room types: ")
listings_df.select("room_type").distinct().show()

print(f"Distinct neighbourhoods: {listings_df.select('neighbourhood_cleansed').distinct().count()}")

Distinct room types: 
+---------------+
|      room_type|
+---------------+
|   Private room|
|    Shared room|
|Entire home/apt|
|     Hotel room|
+---------------+

Distinct neighbourhoods: 33


In [26]:
# Check for nulls
null_counts = listings_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in ["price", "review_scores_rating", "host_name",
             "neighbourhood_cleansed", "room_type"]
])
null_counts.show()

[Stage 52:>                                                         (0 + 1) / 1]

+-----+--------------------+---------+----------------------+---------+
|price|review_scores_rating|host_name|neighbourhood_cleansed|room_type|
+-----+--------------------+---------+----------------------+---------+
|34908|               24122|       43|                     0|        0|
+-----+--------------------+---------+----------------------+---------+



### Data Cleaning & Transformations

In [28]:
# Clean 'price' column
# Price is stored as a string like "$1,234.00" - strip $, comma -> cast to float

listings_clean = listings_df.withColumn(
    "price_usd",
    F.regexp_replace(F.col("price"), "[\\$]", "").cast("float")
)

listings_clean.select("price", "price_usd").show(5)

+-------+---------+
|  price|price_usd|
+-------+---------+
| $70.00|     70.0|
|$149.00|    149.0|
|$411.00|    411.0|
|   NULL|     NULL|
|$210.00|    210.0|
+-------+---------+
only showing top 5 rows



In [36]:
# Handle calendar price + availability
calendar_clean = calendar_df \
    .withColumn(
        "price_usd",
        F.regexp_replace(F.col("price"), "[\\$,]", "").cast("float")
    ) \
    .withColumn(
        "date",
        F.to_date(F.col("date"), "yyyy-MM-dd")
    ) \
    .withColumn(
        "is_available",
        F.when(F.col("available") == "t", 1).otherwise(0)
    )

calendar_clean.select(
    "listing_id",
    "date",
    "available",
    "is_available",
    "price",
    "price_usd"
).show(5)

+-------------------+----------+---------+------------+-----+---------+
|         listing_id|      date|available|is_available|price|price_usd|
+-------------------+----------+---------+------------+-----+---------+
|1196288722069341420|2025-09-15|        f|           0| NULL|     NULL|
|1196288722069341420|2025-09-16|        f|           0| NULL|     NULL|
|1196288722069341420|2025-09-17|        f|           0| NULL|     NULL|
|1196288722069341420|2025-09-18|        f|           0| NULL|     NULL|
|1196288722069341420|2025-09-19|        f|           0| NULL|     NULL|
+-------------------+----------+---------+------------+-----+---------+
only showing top 5 rows



In [39]:
# Parse review date
reviews_clean = reviews_df \
    .withColumn("date", F.to_date(F.col("date"), "yyyy-MM-dd")) \
    .withColumn("year",  F.year("date")) \
    .withColumn("month", F.month("date"))

reviews_clean.select("listing_id", "date", "year", "month", "reviewer_name").show(5)

+----------+----------+----+-----+-------------+
|listing_id|      date|year|month|reviewer_name|
+----------+----------+----+-----+-------------+
|     13913|2010-08-18|2010|    8|      Michael|
|     13913|2011-07-11|2011|    7|      Mathias|
|     13913|2011-09-13|2011|    9|      Kristin|
|     13913|2011-10-03|2011|   10|      Camilla|
|     13913|2011-10-09|2011|   10|        Jorik|
+----------+----------+----+-----+-------------+
only showing top 5 rows



In [46]:
# Add derived columns to listings
listings_clean = listings_clean \
    .withColumn(
        "price_category",
        F.when(F.col("price_usd") < 50, "Budget")
         .when(F.col("price_usd") < 150, "Mid-range")
         .when(F.col("price_usd") < 300, "Premium")
         .otherwise("Luxury")
    ) \
    .withColumn(
        "host_is_superhost_bool",
        F.when(F.col("host_is_superhost") == "t", True).otherwise(False)
    ) \
    .withColumn(
        "last_review_date", F.to_date(F.col("last_review"), "yyyy-MM-dd")
    ) \
    .fillna({"review_scores_rating": 0, "price_usd": 0})


listings_clean.select(
    "name", "price_usd", "price_category", "host_is_superhost_bool"
).show(10, truncate=40)
        

+----------------------------------------+---------+--------------+----------------------+
|                                    name|price_usd|price_category|host_is_superhost_bool|
+----------------------------------------+---------+--------------+----------------------+
|     Holiday London DB Room Let-on going|     70.0|     Mid-range|                  true|
|     Bright Chelsea  Apartment. Chelsea!|    149.0|     Mid-range|                 false|
|Very Central Modern 3-Bed/2 Bath By O...|    411.0|        Luxury|                  true|
|        Battersea live/work artist house|      0.0|        Budget|                 false|
|Bright 1 bedroom apt off brick lane i...|    210.0|       Premium|                 false|
|     Kew Gardens 3BR house in cul-de-sac|    280.0|       Premium|                 false|
|         You are GUARANTEED to love this|     90.0|     Mid-range|                  true|
|SUNNY ROOM PRIVATE BATHROOM PLUS BREA...|     61.0|     Mid-range|                  true|

In [47]:
# Drop duplicate rows(if any)
before = listings_clean.count()
listings_clean = listings_clean.dropDuplicates(["id"])
after = listings_clean.count()
print(f"Removed {before - after} duplicate listings IDs")

Removed 0 duplicate listings IDs


In [48]:
# Rename colummns for clarity
listings_clean = listings_clean \
    .withColumnRenamed("neighbourhood_cleansed", "neighbourhood") \
    .withColumnRenamed("calculated_host_listings_count", "host_total_listings")

print("Cleaned listings columns sample:")
print([c for c in listings_clean.columns[:20]])

Cleaned listings columns sample:
['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_id', 'host_url', 'host_name', 'host_since', 'host_location', 'host_about', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_thumbnail_url']


### Aggregations & GroupBy

In [50]:
listings_clean.groupBy("room_type") \
    .agg(
        F.count("id").alias("total_istings"),
        F.round(F.avg("price_usd"), 2).alias("avg_price"),
        F.round(F.min("price_usd"), 2).alias("min_price"),
        F.round(F.max("price_usd"), 2).alias("max_price")
    ) \
    .orderBy(F.desc("avg_price")) \
    .show()

[Stage 70:>                                                         (0 + 1) / 1]

+---------------+-------------+---------+---------+---------+
|      room_type|total_istings|avg_price|min_price|max_price|
+---------------+-------------+---------+---------+---------+
|Entire home/apt|        62907|   142.32|      0.0|    999.0|
|     Hotel room|          109|   102.94|      0.0|    900.0|
|    Shared room|          212|    51.46|      0.0|    900.0|
|   Private room|        33643|    46.57|      0.0|    999.0|
+---------------+-------------+---------+---------+---------+



In [51]:
listings_clean.columns

['id',
 'listing_url',
 'scrape_id',
 'last_scraped',
 'source',
 'name',
 'description',
 'neighborhood_overview',
 'picture_url',
 'host_id',
 'host_url',
 'host_name',
 'host_since',
 'host_location',
 'host_about',
 'host_response_time',
 'host_response_rate',
 'host_acceptance_rate',
 'host_is_superhost',
 'host_thumbnail_url',
 'host_picture_url',
 'host_neighbourhood',
 'host_listings_count',
 'host_total_listings_count',
 'host_verifications',
 'host_has_profile_pic',
 'host_identity_verified',
 'neighbourhood',
 'neighbourhood',
 'neighbourhood_group_cleansed',
 'latitude',
 'longitude',
 'property_type',
 'room_type',
 'accommodates',
 'bathrooms',
 'bathrooms_text',
 'bedrooms',
 'beds',
 'amenities',
 'price',
 'minimum_nights',
 'maximum_nights',
 'minimum_minimum_nights',
 'maximum_minimum_nights',
 'minimum_maximum_nights',
 'maximum_maximum_nights',
 'minimum_nights_avg_ntm',
 'maximum_nights_avg_ntm',
 'calendar_updated',
 'has_availability',
 'availability_30',
 'avai

In [54]:
# Fix duplicate column names
cols = listings_clean.columns
new_cols = []
seen = {}

for c in cols:
    if c in seen:
        seen[c] += 1
        new_cols.append(f"{c}_{seen[c]}")
    else:
        seen[c] = 0
        new_cols.append(c)

listings_clean = listings_clean.toDF(*new_cols)

listings_clean.groupBy("neighbourhood") \
    .agg(
        F.count("id").alias("num_listings"),
        F.round(F.avg("price_usd"), 2).alias("avg_price"),
        F.round(F.avg("review_scores_rating"), 2).alias("avg_rating")
    ) \
    .orderBy(F.desc("num_listings")) \
    .show(10)

[Stage 76:>                                                         (0 + 1) / 1]

+--------------------+------------+---------+----------+
|       neighbourhood|num_listings|avg_price|avg_rating|
+--------------------+------------+---------+----------+
|                NULL|       55662|   108.56|      3.15|
|Neighborhood high...|       41209|   109.18|      4.02|
+--------------------+------------+---------+----------+



In [55]:
# Super host vs regular host
listings_clean.groupBy("host_is_superhost_bool") \
    .agg(
        F.count("id").alias("count"),
        F.round(F.avg("price_usd"), 2).alias("avg_price"),
        F.round(F.avg("review_scores_rating"), 2).alias("avg_rating"),
        F.round(F.avg("number_of_reviews"), 1).alias("avg_reviews")
    ) \
    .show()

[Stage 82:>                                                         (0 + 1) / 1]

+----------------------+-----+---------+----------+-----------+
|host_is_superhost_bool|count|avg_price|avg_rating|avg_reviews|
+----------------------+-----+---------+----------+-----------+
|                 false|79679|    98.98|      3.31|       15.1|
|                  true|17192|   154.45|      4.47|       52.2|
+----------------------+-----+---------+----------+-----------+



In [56]:
# Review volume per year
reviews_clean.groupBy("year") \
    .agg(F.count("id").alias("total_reviews")) \
    .orderBy("year") \
    .show(20)

[Stage 88:>                                                         (0 + 1) / 1]

+----+-------------+
|year|total_reviews|
+----+-------------+
|2009|            1|
|2010|           87|
|2011|          698|
|2012|         3048|
|2013|         9154|
|2014|        18640|
|2015|        37748|
|2016|        68172|
|2017|        97118|
|2018|       133514|
|2019|       170373|
|2020|        49408|
|2021|        77690|
|2022|       229571|
|2023|       346172|
|2024|       450473|
|2025|       406129|
+----+-------------+



In [57]:
# Calendar: Average availability rate by month
calendar_clean \
    .withColumn("month", F.month("date")) \
    .groupBy("month") \
    .agg(
        F.round(F.avg("is_available") * 100, 2).alias("availability_pct"),
        F.round(F.avg("price_usd"), 2).alias("avg_daily_price")
    ) \
    .orderBy("month") \
    .show(12)

[Stage 91:>                                                         (0 + 1) / 1]

+-----+----------------+---------------+
|month|availability_pct|avg_daily_price|
+-----+----------------+---------------+
|    1|           46.17|           NULL|
|    2|           46.88|           NULL|
|    3|           43.71|           NULL|
|    4|           40.58|           NULL|
|    5|           40.65|           NULL|
|    6|           35.75|           NULL|
|    7|           32.45|           NULL|
|    8|            32.8|           NULL|
|    9|           29.97|           NULL|
|   10|           38.45|           NULL|
|   11|           45.89|           NULL|
|   12|           43.81|           NULL|
+-----+----------------+---------------+



In [58]:
# Price distribution category
listings_clean.groupBy("price_category") \
    .count() \
    .orderBy(
        F.when(F.col("price_category") == "Budget",    1)
         .when(F.col("price_category") == "Mid-range", 2)
         .when(F.col("price_category") == "Premium",   3)
         .otherwise(4)
    ) \
    .show()

[Stage 94:>                                                         (0 + 1) / 1]

+--------------+-----+
|price_category|count|
+--------------+-----+
|        Budget|41834|
|     Mid-range|28333|
|       Premium|18535|
|        Luxury| 8169|
+--------------+-----+



### Joins

In [59]:
# Count reviews per listing, then join to listings
review_counts = reviews_clean \
    .groupBy("listing_id") \
    .agg(
        F.count("id").alias("review_count"),
        F.max("date").alias("most_recent_review")
    )

# Cast listing id to same type for join
listings_with_reviews = listings_clean \
    .join(
        review_counts,
        listings_clean["id"].cast("long") == review_counts["listing_id"],
        how="left"
    ) \
    .fillna({"review_count": 0})

listings_with_reviews.select(
    "id", "name", "room_type", "price_usd", "review_count", "most_recent_review"
).show(10, truncate=40)

+-----+----------------------------------------+---------------+---------+------------+------------------+
|   id|                                    name|      room_type|price_usd|review_count|most_recent_review|
+-----+----------------------------------------+---------------+---------+------------+------------------+
|36274|Bright 1 bedroom apt off brick lane i...|Entire home/apt|    210.0|          15|        2025-09-06|
|38995|SPACIOUS ROOM IN CONTEMPORARY STYLE FLAT|   Private room|     49.0|          72|        2025-07-26|
|41445|2 Double bed apartment in quiet area ...|Entire home/apt|    213.0|          25|        2024-08-24|
|41712|Room with a view, shared flat,  centr...|   Private room|     96.0|         137|        2025-06-29|
|41870|           Room in relaxed family house!|   Private room|      0.0|           2|        2011-07-02|
|42692|       Fabulous flat w garden and bkfst!|   Private room|      0.0|           0|              NULL|
|43202|      Beautiful 1 bed apt in Q

### Join calender with listings to get neighbourhood-level pricing


In [60]:
listings_slim = listings_clean.select(
    F.col("id").cast("long").alias("listing_id_l"),
    "neighbourhood",
    "room_type"
)

calendar_enriched = calendar_clean.join(
    listings_slim,
    calendar_clean["listing_id"] == listings_slim["listing_id_l"],
    how="left"
).drop("listing_id_l")

# Avg daily price per neighbourhood per month
calendar_enriched \
    .withColumn("month", F.month("date")) \
    .groupBy("neighbourhood", "month") \
    .agg(F.round(F.avg("price_usd"), 2).alias("avg_price")) \
    .orderBy("neighbourhood", "month") \
    .show(15)

[Stage 109:>                                                        (0 + 1) / 1]

+--------------------+-----+---------+
|       neighbourhood|month|avg_price|
+--------------------+-----+---------+
|                NULL|    1|     NULL|
|                NULL|    2|     NULL|
|                NULL|    3|     NULL|
|                NULL|    4|     NULL|
|                NULL|    5|     NULL|
|                NULL|    6|     NULL|
|                NULL|    7|     NULL|
|                NULL|    8|     NULL|
|                NULL|    9|     NULL|
|                NULL|   10|     NULL|
|                NULL|   11|     NULL|
|                NULL|   12|     NULL|
|Neighborhood high...|    1|     NULL|
|Neighborhood high...|    2|     NULL|
|Neighborhood high...|    3|     NULL|
+--------------------+-----+---------+
only showing top 15 rows



### Anti-join: Listings that have NEVER been reviewed 

In [61]:
reviewed_ids = reviews_clean.select(
    F.col("listing_id").cast("string")
).distinct()

never_reviewed = listings_clean.join(
    reviewed_ids,
    listings_clean["id"].cast("string") == reviewed_ids["listing_id"],
    how="left_anti"
)
print(f"Listings with zero reviews: {never_reviewed.count():,}")
never_reviewed.select("id", "name", "price_usd", "room_type").show(5, truncate=40)

Listings with zero reviews: 24,122


[Stage 126:>                                                        (0 + 1) / 1]

+------+----------------------------------------+---------+------------+
|    id|                                    name|price_usd|   room_type|
+------+----------------------------------------+---------+------------+
|422095|     Stunning Shared Penthouse Apartment|      0.0|Private room|
|434555|     Well furnished room (Olympics site)|      0.0|Private room|
|471753|     3 MASTER BEDROOMS! READ DESCRIPTION|      0.0|Private room|
|540002|1 bedroom in a 2 bedroom flat in sout...|     92.0|Private room|
|554935|        Notting Hill Penthouse Apartment|      0.0|Private room|
+------+----------------------------------------+---------+------------+
only showing top 5 rows



### Spark SQL

In [62]:
# Register DataFrames as temp SQL views
listings_clean.createOrReplaceTempView("listings")
reviews_clean.createOrReplaceTempView("reviews")
calendar_clean.createOrReplaceTempView("calendar")

print("Temp views registered: listings, reviews, calendar")

Temp views registered: listings, reviews, calendar


In [63]:
# Top 10 highest-rated budget listings
spark.sql("""
    SELECT
        name,
        neighbourhood,
        room_type,
        price_usd,
        review_scores_rating,
        number_of_reviews
    FROM listings
    WHERE price_category = 'Budget'
      AND review_scores_rating >= 4.5
      AND number_of_reviews > 10
    ORDER BY review_scores_rating DESC, number_of_reviews DESC
    LIMIT 10
""").show(truncate=40)

[Stage 134:>                                                        (0 + 1) / 1]

+----------------------------------------+-----------------------+------------+---------+--------------------+-----------------+
|                                    name|          neighbourhood|   room_type|price_usd|review_scores_rating|number_of_reviews|
+----------------------------------------+-----------------------+------------+---------+--------------------+-----------------+
|Serene Surroundings - King Size 1 Bed...|Neighborhood highlights|Private room|     49.0|                 5.0|              284|
|Perfect Stay in a Garden Flat - Great...|Neighborhood highlights|Private room|     46.0|                 5.0|              141|
|      Great double room in Bromley North|Neighborhood highlights|Private room|      0.0|                 5.0|              124|
|Bright and airy double room in Lower ...|Neighborhood highlights|Private room|      0.0|                 5.0|              108|
|Spacious Double Room  Private Shower,...|Neighborhood highlights|Private room|      0.0|        

In [ ]:
# Monthly review treds
spark.sql("""
    SELECT
        year,
        month,
        COUNT(*) AS reviews_count
    FROM reviews
    WHERE year IS NOT NULL
    GROUP BY year, month
    ORDER BY year, month
""").show(24)